In [23]:
import sys
from pathlib import Path

sys.path.insert(0, "../src")

import pandas as pd

from ufa import (
    prepare_all_games_training_data,
    train_etv_models,
    build_etv_model,
    save_model_bundle,
    build_game_throws,
)

In [24]:
training_throws = prepare_all_games_training_data("../data/raw/all_games_1024.csv")

print("Training games:", training_throws["gameID"].nunique())
print("Training throws:", len(training_throws))
print("Completion rate:", training_throws["completion"].mean().round(3))
print("Point outcome rate:", training_throws["point_outcome"].mean().round(3))

training_throws[[
    "gameID",
    "thrower",
    "receiver",
    "completion",
    "point_outcome",
    "throw_distance",
    "times",
]].head(20)

,gameID,awayTeamID,homeTeamID,awayScore,homeScore,status
0,2024-06-01-ATX-HTX,sol,havoc,25,15,Final
1,2024-06-01-CHI-IND,union,alleycats,20,21,Final
2,2024-06-01-SEA-SD,cascades,growlers,18,17,Final
3,2024-06-01-TOR-PHI,rush,phoenix,16,18,Final
4,2024-06-01-CAR-DAL,flyers,legion,21,14,Final
5,2024-06-02-DC-BOS,breeze,glory,15,16,Final
6,2024-06-02-DET-MAD,mechanix,radicals,20,28,Final
7,2024-06-02-IND-PIT,alleycats,thunderbirds,18,26,Final
8,2024-06-02-SEA-LA,cascades,aviators,20,19,Final
9,2024-06-07-ATX-ATL,sol,hustle,16,23,Final


In [25]:
training_throws[[
    "thrower_x",
    "thrower_y",
    "receiver_x",
    "receiver_y",
    "throw_distance",
    "throw_angle",
    "y_diff",
    "x_diff",
    "times",
    "completion_target",
    "eventual_score_target",
]].describe()

Fetching 2024-06-01-ATX-HTX
Fetching 2024-06-01-CHI-IND
Fetching 2024-06-01-SEA-SD
Fetching 2024-06-01-TOR-PHI
Fetching 2024-06-01-CAR-DAL
Fetching 2024-06-02-DC-BOS
Fetching 2024-06-02-DET-MAD
Fetching 2024-06-02-IND-PIT
Fetching 2024-06-02-SEA-LA
Fetching 2024-06-07-ATX-ATL
Fetching 2024-06-07-LA-SEA
Fetching 2024-06-07-COL-OAK
Fetching 2024-06-08-LA-POR
Fetching 2024-06-08-HTX-DAL
Fetching 2024-06-08-ATX-CAR
Fetching 2024-06-08-MIN-IND
Fetching 2024-06-08-MAD-DET
Fetching 2024-06-08-NY-DC
Fetching 2024-06-08-COL-SLC
Fetching 2024-06-08-TOR-MTL
Fetching 2024-06-08-PHI-BOS
Fetching 2024-06-14-COL-MAD
Fetching 2024-06-14-HTX-ATL
Fetching 2024-06-14-PIT-CHI
Fetching 2024-06-15-LA-SD
Fetching 2024-06-15-DC-TOR
Fetching 2024-06-15-NY-PHI
Fetching 2024-06-15-COL-MIN
Fetching 2024-06-15-HTX-CAR
Fetching 2024-06-15-DET-IND
Fetching 2024-06-15-DAL-ATX
Fetching 2024-06-15-SEA-POR
Training games: 32
Training throws: 18140
Training throws with attached defender/block: 0


,gameID,thrower,receiver,defender,turnover
0,2024-06-01-ATX-HTX,mbennett1,NaN,NaN,1
1,2024-06-01-ATX-HTX,badams1,mbaldauf,NaN,0
2,2024-06-01-ATX-HTX,jzuraw,dzeringue,NaN,0
3,2024-06-01-ATX-HTX,mbennett1,jzuraw,NaN,0
4,2024-06-01-ATX-HTX,jzuraw,dbossert,NaN,0
5,2024-06-01-ATX-HTX,dbossert,bgfroerer,NaN,0
6,2024-06-01-ATX-HTX,bgfroerer,mbennett1,NaN,0
7,2024-06-01-ATX-HTX,mbennett1,sseidenbe,NaN,0
8,2024-06-01-ATX-HTX,sseidenbe,blewis,NaN,0
9,2024-06-01-ATX-HTX,blewis,jzuraw,NaN,0


In [26]:
model_bundle = train_etv_models(training_throws)
etv_model = build_etv_model(model_bundle)

In [27]:
Path("../models").mkdir(exist_ok=True)

save_model_bundle(model_bundle, "../models/all_games_baseline_etv_model.joblib")

'../models/baseline_etv_model.joblib'

In [28]:
# Train on many games above, then inspect one held-out/display game here.
display_game_id = "2024-06-08-NY-DC"

result = build_game_throws(
    game_id=display_game_id,
    etv_model=etv_model,
)

result.throws[
    ["thrower", "receiver", "completion", "xcp", "etv", "cpoe", "t_ec", "r_ec", "total_ec"]
].head(20)

,thrower,receiver,completion,xcp,etv,cpoe,t_ec,r_ec,total_ec
5,tedmonds,jtom,1,0.966118,0.014401,0.033882,0.001763,0.016763,0.018527
6,jtom,dbloodgoo,1,0.966654,0.027680,0.033346,0.010917,0.031142,0.042060
7,dbloodgoo,cmccutche,1,0.948402,0.057858,0.051598,0.026716,0.065680,0.092395
8,cmccutche,tedmonds,1,0.970602,0.053759,0.029398,-0.011921,0.057582,0.045661
9,tedmonds,dbloodgoo,1,0.974416,0.042821,0.025584,-0.014760,0.045413,0.030652
10,dbloodgoo,cjurek,1,0.953469,0.081100,0.046531,0.035687,0.088167,0.123854
11,cjurek,tedmonds,1,0.956373,0.077018,0.043627,-0.011149,0.083557,0.072408
12,tedmonds,cjurek,1,0.975169,0.066743,0.024831,-0.016814,0.069938,0.053124
13,cjurek,tedmonds,1,0.963924,0.095416,0.036076,0.025477,0.101540,0.127018
14,tedmonds,cjurek,1,0.781644,0.724534,0.218356,0.622993,1.000000,1.622993


In [29]:
print("Display game:", result.game_id)
print("Throws with attached defender/block:", result.throws["defender"].notna().sum())
print("Box score total blocks:", result.box_score_stats["B"].sum())

result.box_score_stats.sort_values("B", ascending=False).head(40)

Display game: 2024-06-08-NY-DC
Throws with attached defender/block: 0
Box score total blocks: 0.0


,player,G,A,HA,T,B,C,CP%,HuR,HuCP%,...,total_t_ec,avg_t_ec,total_r_ec,avg_r_ec,total_total_ec,avg_total_ec,total_etv,avg_etv,expected_completions,completions_over_expected
0,jrandolph,3.0,3,3.0,0,0.0,36,1.000000,0.055556,1.000000,...,2.505142,0.069587,8.335920,0.231553,10.841062,0.301141,7.346602,0.204072,34.033295,1.966705
1,cjurek,2.0,3,2.0,1,0.0,26,0.962963,0.037037,0.000000,...,2.061624,0.076356,6.254512,0.231649,8.316135,0.308005,6.278301,0.232530,25.583220,0.416780
2,ochartock,4.0,1,2.0,1,0.0,24,0.960000,0.000000,NaN,...,1.614364,0.064575,3.804552,0.152182,5.418916,0.216757,4.374150,0.174966,23.917309,0.082691
3,bjagt,2.0,2,0.0,0,0.0,19,1.000000,0.052632,1.000000,...,1.587667,0.083561,4.390558,0.231082,5.978225,0.314643,3.770511,0.198448,17.951636,1.048364
4,jwilliams,3.0,3,1.0,3,0.0,27,0.900000,0.100000,0.333333,...,2.403592,0.080120,5.163432,0.172114,7.567024,0.252234,5.629139,0.187638,27.649022,-0.649022
5,jmalks,1.0,4,1.0,2,0.0,27,0.931034,0.068966,1.000000,...,2.677275,0.092320,6.048440,0.208567,8.725714,0.300887,5.539171,0.191006,27.405038,-0.405038
6,tedmonds,0.0,3,1.0,0,0.0,24,1.000000,0.083333,1.000000,...,2.085696,0.086904,4.951682,0.206320,7.037378,0.293224,4.103860,0.170994,22.491556,1.508444
7,ebonnet,3.0,1,0.0,1,0.0,20,0.952381,0.000000,NaN,...,1.137657,0.054174,2.134464,0.101641,3.272121,0.155815,2.851960,0.135808,20.225019,-0.225019
8,tholland,2.0,1,0.0,0,0.0,14,1.000000,0.000000,NaN,...,0.568420,0.040601,2.256359,0.161169,2.824779,0.201770,2.081565,0.148683,13.449664,0.550336
9,cmccutche,3.0,0,3.0,0,0.0,8,1.000000,0.000000,NaN,...,0.376824,0.047103,1.523386,0.190423,1.900210,0.237526,1.416432,0.177054,7.682497,0.317503


In [14]:
result.thrower_stats.head(20)

,thrower,attempts,completions,turnovers,huck_attempts,huck_completions,avg_throw_distance,avg_field_y_delta,total_field_y_delta,total_xcp,...,total_total_ec,avg_total_ec,total_etv,avg_etv,expected_completions,completions_over_expected,completion_pct,huck_rate,huck_completion_pct,cpoe
0,aagami,1,1,0,0,0,16.112359,-13.840000,-13.84,0.982590,...,-0.030021,-0.030021,0.048136,0.048136,0.982590,0.017410,1.000000,0.000000,NaN,0.017410
1,adavis2,3,3,0,0,0,17.009500,11.976667,35.93,2.867342,...,0.262956,0.087652,0.162576,0.054192,2.867342,0.132658,1.000000,0.000000,NaN,0.044219
2,amerriman,2,2,0,0,0,17.320931,3.915000,7.83,1.930149,...,0.419635,0.209817,0.276216,0.138108,1.930149,0.069851,1.000000,0.000000,NaN,0.034926
3,aolcese,1,1,0,0,0,22.791974,22.300000,22.30,0.939606,...,0.048093,0.048093,0.027446,0.027446,0.939606,0.060394,1.000000,0.000000,NaN,0.060394
4,aroy,33,31,2,2,0,17.322645,8.071818,266.37,31.247732,...,2.291787,0.069448,2.172205,0.065824,31.247732,-0.247732,0.939394,0.060606,0.0,-0.007507
5,bjagt,19,19,0,1,1,15.158649,7.843684,149.03,17.951636,...,5.978225,0.314643,3.770511,0.198448,17.951636,1.048364,1.000000,0.052632,1.0,0.055177
6,cjurek,27,26,1,1,0,13.890912,6.659630,179.81,25.583220,...,8.316135,0.308005,6.278301,0.232530,25.583220,0.416780,0.962963,0.037037,0.0,0.015436
7,cmccutche,8,8,0,0,0,10.818045,6.161250,49.29,7.682497,...,1.900210,0.237526,1.416432,0.177054,7.682497,0.317503,1.000000,0.000000,NaN,0.039688
8,cseamon,1,1,0,0,0,19.029149,8.590000,8.59,0.973419,...,0.014381,0.014381,0.010314,0.010314,0.973419,0.026581,1.000000,0.000000,NaN,0.026581
9,cweinberg,28,26,2,2,1,18.321000,9.474643,265.29,26.286086,...,7.462989,0.266535,5.317321,0.189904,26.286086,-0.286086,0.928571,0.071429,0.5,-0.010217
